In [1]:
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
URLS = {
    "2019-2022": "https://www.ncsl.org/technology-and-communication/legislation-related-to-artificial-intelligence",
    "2023": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2023-legislation",
    "2024": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2024-legislation",
    "2025": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2025-legislation",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-scraper/1.0)"}

In [ ]:
legi_data = []


def clean(s):
    return re.sub(r"\s+", " ", s).strip()


def get_tables(URL):
    # Get HTML data using BeautifulSoup
    resp = requests.get(URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    # Extract headers and data
    for table in soup.find_all("table"):
        rows = table.find_all("tr")

        if not rows:
            continue

        # First row of table data contains headers
        headers = [clean(c.get_text(" "))
                   for c in rows[0].find_all(["th", "td"])]
        n_cols = len(headers)
        headers.append("URL")

        table_data = []
        for row in rows[1:]:
            data = [clean(c.get_text(" ")) for c in row.find_all(["td", "th"])]

            # First link in the row that has an href ("" if there is none)
            url = next((a["href"] for a in row.find_all("a", href=True)), "")

            if not (any(data) or url):
                continue

            # Pad or trim the cells to the header width, then add the URL later on
            data = (data + [""] * n_cols)[:n_cols]
            data.append(url)
            table_data.append(data)

        legi_data.append(pd.DataFrame(table_data, columns=headers))

In [4]:
# Get each web page into a dataframe by year
for title, url in URLS.items():
    get_tables(url)

In [ ]:
# Add years to each DataFrame
years = [2022, 2021, 2020, 2019, 2023, 2024, 2025]
assert len(legi_data) == len(
    years), f"Expected {len(years)} tables, found {len(legi_data)}"

for i, year_data in enumerate(legi_data):
    legi_data[i]["Page Year"] = years[i]
    # print(years[i], "-->", list(legi_data[i].columns))

# Change column titles to be uniform
legi_data[5] = legi_data[5].rename({"Jurisdiction": "State"}, axis=1)
legi_data[6] = legi_data[6].rename(
    {"Jurisdiction and Summary": "State"}, axis=1)

In [6]:
assert len(legi_data) == 7

# Combine and export data to a csv
legi_data_df = pd.concat(legi_data)
legi_data_df.to_csv("data/ncsl_ai_data_base.csv", index=False)